# Download Science 2022 FRB Polarization Data

Paper: **Frequency-dependent polarization of repeating fast radio bursts - implications for their origin** (Science, 2022).

The paper's Data Availability points to Science Data Bank DOI `10.11922/sciencedb.o00069.00006` and the analysis code repository `https://github.com/SukiYume/RMS`.

This notebook downloads the public ScienceDB V5 data files into `./data/science2022/files/`, preserving the ScienceDB directory layout. The files are PSRCHIVE-like `.calibP` dynamic-spectrum / polarization products containing Stokes I, Q, U, V.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs, unquote
from collections import Counter
import json
import os
import shlex
import subprocess

ROOT = Path('..').resolve() if Path.cwd().name == 'scripts' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'science2022'
FILES_DIR = DATA_DIR / 'files'
DATASET_ID = 'ad2750be4335496f9766e386e4e5f1b1'
VERSION = 'V5'
DOI = '10.11922/sciencedb.o00069.00006'

DATA_DIR.mkdir(parents=True, exist_ok=True)
FILES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR

## 1. Save Metadata

The DOI currently resolves to:

`https://www.scidb.cn/en/detail?dataSetId=ad2750be4335496f9766e386e4e5f1b1`

ScienceDB exposes Croissant metadata through a public API.

In [ ]:
detail_url = f'https://www.scidb.cn/detail?dataSetId={DATASET_ID}'
croissant_url = f'https://www.scidb.cn/api/gin-sdb-croissant/public/getCroissant?datasetId={DATASET_ID}&version={VERSION}'

subprocess.run(['curl', '-L', detail_url, '-o', str(DATA_DIR / 'scidb_detail.html')], check=True)
subprocess.run(['curl', '-L', croissant_url, '-o', str(DATA_DIR / f'croissant_{VERSION}.json')], check=True)

with (DATA_DIR / f'croissant_{VERSION}.json').open() as f:
    metadata = json.load(f)

dataset = metadata['data']
print(dataset['name'])
print(dataset['identifier'])
print(dataset['size'])
print(dataset['description'])

## 2. Get Download URL List

ScienceDB's frontend uses the public endpoint below to export all per-file download URLs. The `global` value is only used by the frontend as a timezone/city hint; `Shanghai` is fine here.

In [ ]:
all_url_path = DATA_DIR / f'all_url_{VERSION}.txt'
all_url_api = (
    'https://www.scidb.cn/api/sdb-filetree-service/getAllUrl?'
    f'dataSetId={DATASET_ID}&version={VERSION}&global=Shanghai'
)
subprocess.run(['curl', '-L', all_url_api, '-o', str(all_url_path)], check=True)

urls = [line.strip() for line in all_url_path.read_text().splitlines() if line.strip()]
print('num files:', len(urls))
print('\n'.join(urls[:5]))

In [ ]:
def science_db_relpath(url: str) -> Path:
    query = parse_qs(urlparse(url).query)
    return Path(unquote(query['path'][0]).lstrip('/'))

relpaths = [science_db_relpath(url) for url in urls]
by_group = Counter(path.parts[1] for path in relpaths if len(path.parts) > 1)
print(by_group)
for path in relpaths:
    print(path)

## 3. Download Files

The whole V5 dataset is about 3.7 GB. Downloads use `curl -C -`, so rerunning this cell resumes partial files. Files are saved under `data/science2022/files/V5/...`.

In [ ]:
def download_one(url: str, index: int, total: int) -> Path:
    relpath = science_db_relpath(url)
    out = FILES_DIR / relpath
    out.parent.mkdir(parents=True, exist_ok=True)
    print(f'[{index}/{total}] {relpath}')
    subprocess.run(
        [
            'curl', '-L', '-C', '-',
            '--retry', '5', '--retry-delay', '5',
            url, '-o', str(out),
        ],
        check=True,
    )
    return out

downloaded = []
for i, url in enumerate(urls, 1):
    downloaded.append(download_one(url, i, len(urls)))

print('downloaded files:', len(downloaded))

## 4. Optional: Clone Analysis Code

The paper also lists the code repository `https://github.com/SukiYume/RMS`. Clone it under `data/science2022/RMS` if you want the original analysis scripts. This is optional for downloading the data files.

In [ ]:
repo_dir = DATA_DIR / 'RMS'
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SukiYume/RMS', str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=False)

## Manual Fallback

If CLI download fails, open this page in a browser:

`https://www.scidb.cn/en/detail?dataSetId=ad2750be4335496f9766e386e4e5f1b1`

Use the page's **Get all links** or **Download ZIP** button. If you download manually, place files under:

`data/science2022/files/V5/`